# image segmentation in python using pixel classification for all 4 channels
a notebook to learn image segmentation using labeled images

in Colab, we need to connect to a GPU

on the top right, go to change runtime type and select T4 GPU

In [ ]:
pip install scikit-image

In [ ]:
pip install matplotlib

It is normal that in colab, you might neeed to restart the session when you attempt to pip install apoc.... Simply confirm restart of the session and start from the first executable cell

In [ ]:
!pip install apoc --no-deps
!pip install "scikit-learn" "pyclesperanto-prototype" "pandas"
!pip install "numpy==2.4.4"

In [ ]:
from matplotlib import pyplot as plt
import apoc
from skimage.io import imread, imsave
import numpy as np

In [ ]:
import os
# Clone the repo if not already in Colab
if 'google.colab' in str(get_ipython()):
    if not os.path.exists('/content/NBCimageAnalysis'):
        !git clone https://github.com/FilLieb/NBCimageAnalysis.git
    os.chdir('/content/NBCimageAnalysis/learning/')
    print(os.listdir('.'))

We have 4 channels and assign their names to a list:

In [ ]:
channels = ["DAAO", "Cre", "vGAT", "Gphn"]  # change index to switch between channels

let's check that is all correct...

In [ ]:
for channel in channels: 
    image_folder = '../training_4_channels/' + channel + '/images/'
    masks_folder = '../training_4_channels/' + channel + '/masks/'

    image_path = os.path.join(image_folder, os.listdir(image_folder)[0])
    image = imread(image_path)

    masks_path = os.path.join(masks_folder, os.listdir(masks_folder)[0])
    masks = imread(masks_path)


    f, a = plt.subplots(1,3, figsize=(15,5))
    a[0].imshow(image, cmap='gray')
    a[0].set_title("Image" + channel)

    a[1].imshow(masks, vmin=0, vmax=2)
    a[1].set_title("Masks" + channel)

    a[2].imshow(image, cmap='gray')
    a[2].contour(masks, colors='r', linewidths=0.5)
    a[2].set_title("Overlay" + channel)

plt.show()

we can now loop through this list to train all models...

In [ ]:
for channel in channels: 
    image_folder = '../training_4_channels/' + channel + '/images/'
    masks_folder = '../training_4_channels/' + channel + '/masks/'

     # this is where the model will be saved
    cl_filename = '../training_4_channels/' + channel + '/models/' + channel + '_object_model.cl'
    
    apoc.erase_classifier(cl_filename) # delete it if it was existing before

    # setup classifier and where it should be saved
    segmenter = apoc.ObjectSegmenter(opencl_filename=cl_filename,
                                     max_depth=5,
                                     num_ensembles=1000)

    # setup feature set used for training
    features = apoc.PredefinedFeatureSet.small_dog_log.value + " " + \
               apoc.PredefinedFeatureSet.medium_dog_log.value + " " + \
               apoc.PredefinedFeatureSet.large_dog_log.value

    # train classifier on folders
    apoc.erase_classifier(cl_filename)
    apoc.train_classifier_from_image_folders(
        segmenter,
        features,
        image = image_folder,
        ground_truth = masks_folder)
    
    print("Training completed and model saved to " + cl_filename)



finally we can test how the model performed

In [ ]:
for channel in channels: 
    image_folder = '../training_4_channels/' + channel + '/images/'

    image_path = os.path.join(image_folder, os.listdir(image_folder)[0])
    image = imread(image_path)

    segmenter = apoc.ObjectSegmenter(opencl_filename=cl_filename)
    labels = segmenter.predict(image)

    f, a = plt.subplots(1,3, figsize=(15,5))
    a[0].imshow(image, cmap='gray')
    a[0].set_title("Image" + channel)

    a[1].imshow(labels, vmin=0, vmax=2)
    a[1].set_title("Labels" + channel)

    a[2].imshow(image, cmap='gray')
    a[2].contour(labels, colors='r', linewidths=0.5)
    a[2].set_title("Overlay" + channel)

plt.show()